# Getting Started with SocialMapper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/01-getting-started.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

- Install SocialMapper and verify the installation
- Understand core concepts: isochrones, POIs, and census data
- Use demo mode for quick testing without API keys
- Run your first accessibility analysis
- Handle common errors gracefully

## Installation

Install SocialMapper with routing support for fast isochrones.

In [ ]:
# Install SocialMapper (pin versions for Colab compatibility)
!pip install -q "socialmapper[routing]" "pandas<3.0" "numpy<2.0"

# IMPORTANT: After install, go to Runtime > Restart session, then skip this cell

### Verify Installation

Always verify your installation by checking the version number.

In [ ]:
import socialmapper

print(f"SocialMapper v{socialmapper.__version__}")
print("Installation successful!")

## Demo Mode

SocialMapper includes a **demo mode** that lets you explore features without API keys. This is perfect for:

- Learning the library
- Testing in CI/CD pipelines
- Quick prototyping

> **Tip:** Demo mode uses pre-cached data for select locations (Portland OR, Chapel Hill NC, Durham NC). For production analysis with real data, you'll need API keys.

In [ ]:
import os

# Enable demo mode for this tutorial
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

print("Demo mode enabled!")

### Exploring Demo Functions

The `demo` module provides pre-built examples to help you understand SocialMapper's capabilities.

In [ ]:
from socialmapper import demo

# See all available demo locations
demo.list_available_demos()

In [ ]:
# Quick start demo - see what SocialMapper can do in one call
result = demo.quick_start(location="Portland, OR", travel_time=15)

print(f"Location: {result['location']}")
print(f"Isochrone area: {result['area_sq_km']:.2f} km²")
print(f"POIs found: {result['poi_count']}")
print(f"Total population: {result['total_population']:,}")
print(f"Median income: ${result['median_income']:,}")

In [ ]:
# Specialized demo: library access
library_result = demo.show_libraries(location="Portland, OR")

print(f"Libraries found: {library_result['library_count']}")
print(f"Population served: {library_result['population_served']:,}")
print(f"People per library: {library_result['people_per_library']:,}")

In [ ]:
# Specialized demo: food access
food_result = demo.show_food_access(location="Portland, OR")

print(f"Grocery stores: {food_result['grocery_count']}")
print(f"Restaurants: {food_result['restaurant_count']}")
print(f"Population with food access: {food_result['population_served']:,}")

## Core Concepts

SocialMapper provides five main functions for accessibility analysis:

| Function | Purpose | Returns |
|----------|---------|--------|
| `create_isochrone()` | Generate travel-time polygons | GeoJSON Feature |
| `get_poi()` | Find points of interest | List of POI dicts |
| `get_census_blocks()` | Get census geography | List of block dicts |
| `get_census_data()` | Retrieve demographics | CensusDataResult |
| `create_map()` | Create visualizations | MapResult |

In [ ]:
# Import all main functions
from socialmapper import (
    create_isochrone,
    get_poi,
    get_census_blocks,
    get_census_data,
    create_map
)

print("SocialMapper functions imported successfully!")

## Your First Isochrone

An **isochrone** (from Greek: iso = equal, chronos = time) is a polygon showing all areas reachable from a starting point within a certain travel time.

> **Note:** Unlike simple radius-based buffers, isochrones account for the actual road network, making them much more accurate for accessibility analysis.

In [ ]:
# Create a 15-minute driving isochrone
isochrone = create_isochrone(
    location="Portland, OR",
    travel_time=15,
    travel_mode="drive"
)

# Explore the result
print(f"Isochrone type: {isochrone['type']}")
print(f"Travel time: {isochrone['properties']['travel_time']} minutes")
print(f"Travel mode: {isochrone['properties']['travel_mode']}")
print(f"Area covered: {isochrone['properties']['area_sq_km']:.2f} km²")
print(f"Backend used: {isochrone['properties']['backend']}")

## Finding Points of Interest

Use `get_poi()` to find amenities like hospitals, schools, grocery stores, and more.

In [ ]:
# Find healthcare facilities near Portland
healthcare = get_poi(
    location="Portland, OR",
    categories=["healthcare"],
    limit=10
)

print(f"Found {len(healthcare)} healthcare facilities:")
for h in healthcare[:5]:
    print(f"  - {h['name']}: {h['distance_km']:.2f} km away")

## Getting Census Data

Retrieve demographic information to understand who lives in your study area.

In [ ]:
# Get census blocks within the isochrone
blocks = get_census_blocks(polygon=isochrone)
print(f"Found {len(blocks)} census block groups")

# Get population data
geoids = [b['geoid'] for b in blocks]
census_result = get_census_data(
    location=geoids,
    variables=["population"]
)

# Calculate total population (handling None values)
total_pop = sum(
    data.get("population", 0) or 0
    for data in census_result.data.values()
)
print(f"Total population in area: {total_pop:,}")

## Complete Analysis Example

Let's put it all together: analyze grocery store access for an area.

In [ ]:
# Define our study area
location = "Portland, OR"

print(f"Analyzing grocery access in {location}...")
print("=" * 50)

# Step 1: Create walking isochrone (15 minutes)
walk_area = create_isochrone(
    location=location,
    travel_time=15,
    travel_mode="walk"
)
print(f"\n1. Walking area: {walk_area['properties']['area_sq_km']:.2f} km²")

# Step 2: Find grocery stores
groceries = get_poi(
    location=location,
    categories=["shopping"],
    travel_time=15,
    limit=50
)
print(f"2. Grocery stores found: {len(groceries)}")

# Step 3: Get census data
blocks = get_census_blocks(polygon=walk_area)
geoids = [b['geoid'] for b in blocks]
census = get_census_data(geoids, variables=["population", "median_income"])

# Step 4: Calculate statistics (with robust None handling)
total_pop = sum(
    d.get("population", 0) or 0
    for d in census.data.values()
)
incomes = [
    d["median_income"]
    for d in census.data.values()
    if d.get("median_income") and d["median_income"] > 0
]

print(f"3. Population with walkable access: {total_pop:,}")
if incomes:
    print(f"4. Average median income: ${sum(incomes)/len(incomes):,.0f}")

# Summary
print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Location: {location}")
print(f"Travel mode: 15-minute walk")
print(f"Grocery stores accessible: {len(groceries)}")
print(f"Population served: {total_pop:,}")
if len(groceries) >= 3:
    print("Assessment: Good grocery access")
else:
    print("Assessment: Limited grocery access - potential food desert")

## Working with Coordinates

You can use latitude/longitude coordinates instead of place names.

In [ ]:
# Using coordinates (lat, lon)
portland_coords = (45.5152, -122.6784)

isochrone = create_isochrone(
    location=portland_coords,
    travel_time=10,
    travel_mode="walk"
)

print(f"10-minute walk from coordinates:")
print(f"  Area: {isochrone['properties']['area_sq_km']:.2f} km²")

## Error Handling

SocialMapper provides specific exception classes for different error types. Always wrap API calls in try/except blocks for robust code.

> **Tip:** The exception's `help_text` attribute often contains useful troubleshooting suggestions.

In [ ]:
from socialmapper import (
    SocialMapperError,
    ValidationError,
    InvalidLocationError,
    InvalidPOICategoryError,
    MissingAPIKeyError,
    NetworkError,
    RateLimitError
)

# Example: Handle invalid location gracefully
try:
    isochrone = create_isochrone(
        location="Nonexistent Place, XY",
        travel_time=15
    )
except InvalidLocationError as e:
    print(f"Location error: {e}")
    if e.help_text:
        print(f"Suggestion: {e.help_text}")
except ValidationError as e:
    print(f"Validation error: {e}")
except SocialMapperError as e:
    print(f"General error: {e}")

In [ ]:
# Example: Handle invalid POI category
try:
    pois = get_poi(
        location="Portland, OR",
        categories=["invalid_category"]
    )
except InvalidPOICategoryError as e:
    print(f"Invalid category: {e}")
    print(f"\nValid categories include:")
    for cat in e.valid_categories[:5]:
        print(f"  - {cat}")

## Troubleshooting

### Common Issues and Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| `ModuleNotFoundError` | Missing dependencies | Run `pip install socialmapper[routing]` |
| `MissingAPIKeyError` | Census API key not set | Get free key from census.gov (see below) |
| `InvalidLocationError` | Location not recognized | Use coordinates or check spelling |
| `NetworkError` | Internet connectivity | Check connection, try again |
| `RateLimitError` | Too many API requests | Wait and retry, or use demo mode |

### Getting API Keys

For production use, you'll need:

1. **Census API Key** (required for real census data):
   - Get a free key at: https://api.census.gov/data/key_signup.html
   - Set it: `os.environ["CENSUS_API_KEY"] = "your-key"`

2. **Routing API Keys** (optional, for faster isochrones):
   - OpenRouteService: https://openrouteservice.org/dev/
   - GraphHopper: https://www.graphhopper.com/

> **Note:** The default Valhalla backend doesn't require an API key.

In [ ]:
# Check your current configuration
import os

print("Current Configuration:")
print(f"  Demo mode: {os.environ.get('SOCIALMAPPER_DEMO_MODE', 'false')}")
print(f"  Census API key: {'set' if os.environ.get('CENSUS_API_KEY') else 'not set'}")
print(f"  Routing backend: {os.environ.get('SOCIALMAPPER_ROUTING_BACKEND', 'auto (default)')}")

## Disable Demo Mode for Production

When you're ready to analyze real data, disable demo mode and set your API keys.

In [ ]:
# Disable demo mode for production use
# os.environ.pop("SOCIALMAPPER_DEMO_MODE", None)
# os.environ["CENSUS_API_KEY"] = "your-key-here"

print("For production use:")
print("1. Get a Census API key from: https://api.census.gov/data/key_signup.html")
print("2. Set it as CENSUS_API_KEY environment variable")
print("3. Remove SOCIALMAPPER_DEMO_MODE from environment")

## Next Steps

You've learned the basics of SocialMapper! Continue with:

1. **[Isochrone Analysis](02-isochrone-analysis.ipynb)** - Deep dive into travel-time analysis and routing backends
2. **[Points of Interest](03-points-of-interest.ipynb)** - Advanced POI queries and category filtering
3. **[Census Data](04-census-data.ipynb)** - Working with demographics and data quality
4. **[Mapping & Visualization](05-mapping-visualization.ipynb)** - Creating publication-ready maps
5. **[Complete Workflow](06-complete-workflow.ipynb)** - End-to-end analysis examples
6. **[Food Desert Case Study](07-food-desert-case-study.ipynb)** - Real-world equity analysis